# Experimento individual e baseline

POC DEMO — contrato PROVISÓRIO / PENDENTE DE HOMOLOGAÇÃO. Execute em kernel novo. Credenciais somente por ambiente.


Configuração tipada. Pré-requisitos: Docker saudável, seed DEMO, token por ambiente. Entradas: parâmetros abaixo; saídas: tabelas, IDs e relatório JSON/Markdown. Escritas exigem confirmação EXECUTE_DEMO e chave estável; repetir chave igual recupera o mesmo job. Espere READY; erros de dependência exigem verificar docker compose logs.

In [1]:
import os, json
import pandas as pd
from IPython.display import display, Markdown
from src.workbench.journey_config import Settings
cfg=Settings.load()
from src.workbench.notebook_config import Parameters
from src.workbench.operations_sdk import call, report, await_job
params=Parameters.load()
display(params.model_dump(mode="json"))
display({"state":"READY", "write_enabled":params.execute_write, "tracking":"remote Docker service", "browser_ui":cfg.ui})
evidence={}
NOTEBOOK_ID="04_treinamento_individual"


{'mode': 'READ_EXISTING',
 'proposal_id': None,
 'snapshot_id': 'fb931e1e-43d0-5ff8-938c-0d2ba4a68794',
 'model_name': 'hi.demo.notebook.abdb8f09',
 'model_version': '',
 'campaign_id': None,
 'approval_id': None,
 'execute_write': True,
 'confirmation': 'EXECUTE_DEMO',
 'request_key': 'notebooks-abdb8f09',
 'requester': 'scientist-demo',
 'algorithm': 'ridge',
 'alpha': 1.0,
 'capacity': 40,
 'model_count': 30,
 'parallel_jobs': 2,
 'campaign_action': 'observe',
 'governance_action': 'read',
 'review_action': 'APPROVE',
 'reason': '',
 'lineage_kind': 'proposal',
 'external_id': '',
 'reset_mode': 'preview',
 'reinitialize': 'demo-ready',
 'reset_plan_id': None}

{'state': 'READY',
 'write_enabled': True,
 'tracking': 'remote Docker service',
 'browser_ui': 'http://127.0.0.1:5000'}

1. Selecionar snapshot, algoritmo ridge/dummy e alpha. Uma campanha com um item mantém idempotência e execução independente do kernel.

In [2]:
from src.data.journey import extract
_,snapshot=extract(cfg);display(snapshot)
model_keys=[params.model_name]
display({"model":params.model_name,"algorithm":params.algorithm,"alpha":params.alpha,"baseline":"DummyRegressor mean"})


{'id': 'fb931e1e-43d0-5ff8-938c-0d2ba4a68794',
 'snapshot_sha256': '7b4120a14004aeb7cc006c0e210f004a0a6dffd9510bf304f27641b76dbd377e',
 'cycle_id': '0fa6938d-2743-56ae-a057-dffd2274d90a',
 'target_index': 4052,
 'target_semester': 1,
 'dataset_sha256': '5fc39eeeead59aa48813182e6a93d2308a4a9243655501f17d09a981c6c66a56',
 'query_sha256': 'defc10d1b6ea752600af271285c54107663e91db9df13914aea7e42613c83adf',
 'query_version': 'journey-history-v1'}

{'model': 'hi.demo.notebook.abdb8f09',
 'algorithm': 'ridge',
 'alpha': 1,
 'baseline': 'DummyRegressor mean'}

2. Submeter e acompanhar treino real opt-in; candidato não é promovido.

In [3]:
from src.workbench.operations_sdk import campaign
if params.execute_write:
    params.authorize()
    if not params.snapshot_id:
        raise ValueError("Selecione snapshot_id da tabela SQL")
    payload=dict(snapshot_id=str(params.snapshot_id),cycle_code=cfg.cycle,model_keys=model_keys,
        algorithm=params.algorithm,alpha=params.alpha,max_parallel_jobs=params.parallel_jobs,
        requested_by=params.requester,confirm_training=True)
    display({"preflight":payload,"budget":"máximo 2 processos, 2 GiB no container worker; identidades experimentais sintéticas"})
    submitted=campaign(cfg,payload,params.request_key)
    campaign_id=submitted["campaign_id"]
else:
    campaign_id=str(params.campaign_id) if params.campaign_id else ""
if campaign_id:
    state=await_job(cfg,f"/v1/training/campaigns/{campaign_id}",timeout=7200)
    items=call(cfg,"GET",f"/v1/training/campaigns/{campaign_id}/items?limit=500")
    display(state);display(pd.DataFrame(items))
    evidence.update(campaign=state,items=items)
else:
    evidence["training"]="NOT_EXECUTED: escolha campanha existente ou habilite escrita"


{'preflight': {'snapshot_id': 'fb931e1e-43d0-5ff8-938c-0d2ba4a68794',
  'cycle_code': '2026-1',
  'model_keys': ['hi.demo.notebook.abdb8f09'],
  'algorithm': 'ridge',
  'alpha': 1,
  'max_parallel_jobs': 2,
  'requested_by': 'scientist-demo',
  'confirm_training': True},
 'budget': 'máximo 2 processos, 2 GiB no container worker; identidades experimentais sintéticas'}

{'status': 'COMPLETED', 'counts': {'COMPLETED': 1}, 'utc': '2026-11-04T16:29:15.366619+00:00'}


{'campaign_id': '00b98583-e68e-4174-aebc-d7cf2dd03960',
 'idempotency_key': 'notebooks-abdb8f09',
 'request_hash': '3f110e87ec324e955a892c62d33f5efd07f7a1bccfc9f57f611b045041d2c5e2',
 'request': {'alpha': 1.0,
  'algorithm': 'ridge',
  'cycle_code': '2026-1',
  'model_keys': ['hi.demo.notebook.abdb8f09'],
  'snapshot_id': 'fb931e1e-43d0-5ff8-938c-0d2ba4a68794',
  'max_attempts': 3,
  'requested_by': 'scientist-demo',
  'schema_version': 'poc-training-campaign-v2',
  'confirm_training': True,
  'max_parallel_jobs': 2},
 'correlation_id': '053f60f5-3198-4212-9720-b92ee91ec3b4',
 'requested_by': 'scientist-demo',
 'status': 'COMPLETED',
 'max_parallel_jobs': 2,
 'requested_count': 1,
 'created_at_utc': '2026-11-04T16:29:09.226678+00:00',
 'started_at_utc': '2026-11-04T16:29:09.343239+00:00',
 'ended_at_utc': '2026-11-04T16:29:10.507134+00:00',
 'counts': {'COMPLETED': 1}}

,job_id,campaign_id,model_key,status,attempt,heartbeat_at_utc,started_at_utc,ended_at_utc,mlflow_run_id,registered_model_name,model_version,error_code,result
0,4e63488e-a440-4fed-babe-052716fac7cc,00b98583-e68e-4174-aebc-d7cf2dd03960,hi.demo.notebook.abdb8f09,COMPLETED,1,2026-11-04T16:29:10.238311+00:00,2026-11-04T16:29:09.343239+00:00,2026-11-04T16:29:10.503187+00:00,eb44663c5112462ea5b95fd92c204151,hi.demo.notebook.abdb8f09,1,None,"{'run_id': 'eb44663c5112462ea5b95fd92c204151',..."


3. Comparar métricas, prova de download/checksum e reload feitas pelo job. Se executado, o resultado deve ter três evidências de arquivo por modelo.

In [4]:
for item in evidence.get("items",[]):
    if item["status"]=="COMPLETED":
        display(item["result"])
        assert len(item["result"]["storage_evidence"])==3
        display(Markdown(f"[Run]({cfg.ui}/#/experiments/{item['result']['experiment_id']}/runs/{item['mlflow_run_id']})"))


{'run_id': 'eb44663c5112462ea5b95fd92c204151',
 'metrics': {'mae': 3.758630658273197,
  'rmse': 4.9142272796917865,
  'cpu_seconds': 0.21726249399999986,
  'baseline_mae': 7.466666666666665,
  'peak_rss_kib': 276812.0,
  'baseline_rmse': 8.821497143283464,
  'duration_seconds': 0.526552208000794},
 'model_uri': 'models:/hi.demo.notebook.abdb8f09/1',
 'model_name': 'hi.demo.notebook.abdb8f09',
 'artifact_uri': 'models:/m-3c632036203740d982f1b29aa117263d',
 'experiment_id': '1',
 'model_version': '1',
 'artifact_sha256': '4268666504b258b661b9c6cc009bdf8c21d06fde4317fb389a585e633be3ad0b',
 'storage_evidence': [{'key': 'mlflow/1/models/m-3c632036203740d982f1b29aa117263d/artifacts/MLmodel',
   'bytes': 1336,
   'bucket': 'pucpr-poc',
   'sha256': 'c1456acc4e87b4087686717f5612d0a412cdf9fca4078a9229aed6d272fbc184',
   'status': 'PASS',
   'model_version': '1'},
  {'key': 'mlflow/1/models/m-3c632036203740d982f1b29aa117263d/artifacts/model.pkl',
   'bytes': 8671,
   'bucket': 'pucpr-poc',
   's

[Run](http://127.0.0.1:5000/#/experiments/1/runs/eb44663c5112462ea5b95fd92c204151)

Guardar evidência desta execução e links. Esperado: caminho do relatório, sem credenciais.

In [5]:
display(report(NOTEBOOK_ID,evidence,limitations=["Dados DEMO e contrato provisório; regras não avaliadas não são homologação"]))


'/reports/v2/04_treinamento_individual.json'